In [65]:
# import filtered_combined_military_time.csv

import pandas as pd
massdot = pd.read_csv('../cleaned_data/ACLU_merged_cleaned.csv')
massdot

,Agency Code,Issuing Agency,Officer Id,Citation #,Type,Event/Written Date,Time-HH,Time-MM,AM-PM,Location Name,Offense,Offense Description,Race,Gender,Military Time
0,PD_BPH,Boston Police District C-11,131215,608180AA,WARN,2020-01-01,7,10,AM,Boston,8980,"YIELD AT INTERSECTION, FAIL * c89 §8",ASIAN,MALE,07:10
1,CP_BPX,Boston Police Special OPS,11735,R8170781,CIVIL,2020-01-01,12,45,AM,Boston,894A0,MARKED LANES VIOLATION * c89 §4A,MIDEST,MALE,00:45
2,CP_BPX,Boston Police Special OPS,11735,R8170782,CIVIL,2020-01-01,1,25,AM,Jamaica Plain,8940,"KEEP RIGHT ON HILL/OBSTRUCTED VIEW, FL * c89 §4",BLACK,MALE,01:25
3,PD_BPL,Boston Police District E-18,108874,T1573831,WARN,2020-01-01,4,45,PM,W Roxbury,907F0,MOTORCYCLE PASSENGER VIOLATION * c90 §7,BLACK,FEMALE,16:45
4,PD_BPC,Boston Police District B-3,11695,T1579403,CIVIL,2020-01-01,12,45,PM,Dorchester,909B0,UNREGISTERED MOTOR VEHICLE * c90 §9,BLACK,MALE,12:45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105822,PD_BPB,Boston Police District B-2,106745,T3090876,WARN,2022-12-31,1,55,PM,Roxbury,9018A2,SPEEDING IN VIOL SPECIAL REGULATION * c90 §18,WHITE,FEMALE,13:55
105823,PD_BPB,Boston Police District B-2,106745,T3090877,WARN,2022-12-31,2,0,PM,Roxbury,9018A2,SPEEDING IN VIOL SPECIAL REGULATION * c90 §18,BLACK,MALE,14:00
105824,PD_BPB,Boston Police District B-2,106745,T3090878,WARN,2022-12-31,2,4,PM,Roxbury,9018A2,SPEEDING IN VIOL SPECIAL REGULATION * c90 §18,HISP,FEMALE,14:04
105825,PD_BPB,Boston Police District B-2,106745,T3090879,WARN,2022-12-31,12,0,PM,Roxbury,9018A2,SPEEDING IN VIOL SPECIAL REGULATION * c90 §18,HISP,FEMALE,12:00


In [66]:
# Import the Court_merged_cleaned.csv file
court = pd.read_csv('../cleaned_data/Court_merged_cleaned.csv')
court

,IDNO6,NAME,RANK,ASSIGNED_DESC,CHARGED_DESC,OTDATE,STARTTIME,ENDTIME,WRKDHRS
0,93855.0,"Rodrigues,Joao C",Ptl,YOUTH VIOLENCE STRIKE FORCE,YOUTH VIOLENCE STRIKE FORCE,1/3/2020,830,930,1.00
1,11894.0,"Cutone,Anthony",Det,D-14 DETECTIVES,D-14 DETECTIVES,1/2/2020,1000,1030,0.50
2,131212.0,"O'Loughlin,James M",Ptl,DISTRICT 02,DISTRICT 02,1/3/2020,900,1030,1.50
3,131218.0,"Smerz,Christopher",Ptl,DISTRICT 02,DISTRICT 02,1/3/2020,830,900,0.50
4,103632.0,"Finn,Kevin P",Det,C-11 DETECTIVES,C-11 DETECTIVES,1/3/2020,1530,1600,0.50
...,...,...,...,...,...,...,...,...,...
39745,9823.0,"Morrissey,Sheila M",Civili,OPERATIONS DIVISION,OPERATIONS DIVISION,3/24/2022,830,1215,3.75
39746,108997.0,"Savicke,Erik",Civili,LATENT PRINTS UNIT,LATENT PRINTS UNIT,7/27/2022,1500,1700,2.00
39747,99103.0,"Morgan,Kristen A.",Civili,LATENT PRINTS UNIT,LATENT PRINTS UNIT,8/16/2022,1500,1600,1.00
39748,122348.0,"Xavier,Briana",Civili,OPERATIONS DIVISION,OPERATIONS DIVISION,10/19/2022,930,1100,1.50


In [67]:
# Change "IDNO6" from a float64 to an object 8645.0 to 8645 using strip
court['IDNO6'] = court['IDNO6'].astype(str)
court['IDNO6'] = court['IDNO6'].apply(lambda x: x.strip('.0'))

In [68]:
# Convert the "STARTTIME" column in records to military time like 00:00
court['STARTTIME'] = court['STARTTIME'].astype(str)
court['Military Start Time'] = court['STARTTIME'].apply(lambda x: '00:00' if x == '0' else f"{x.zfill(4)[:2]}:{x.zfill(4)[2:]}")

# Convert the "ENDTIME" column in records to military time like 00:00
court['ENDTIME'] = court['ENDTIME'].astype(str)
court['Military End Time'] = court['ENDTIME'].apply(lambda x: '00:00' if x == '0' else f"{x.zfill(4)[:2]}:{x.zfill(4)[2:]}")

In [69]:
massdot['Event/Written Date'] = pd.to_datetime(massdot['Event/Written Date'])
court['OTDATE'] = pd.to_datetime(court['OTDATE'])

print(court['OTDATE'][0])
print(massdot['Event/Written Date'][0])
print(court['Military Start Time'][0])
print(massdot['Military Time'][0])
print(court['Military End Time'][0])

court['OTDATE'][0] == massdot['Event/Written Date'][300]

2020-01-03 00:00:00
2020-01-01 00:00:00
08:30
07:10
09:30


True

In [70]:
import pandas as pd

# Step 1: Perform a merge to align records based on date and Officer Id
merged_df = pd.merge(
    massdot,
    court,
    left_on=['Event/Written Date', 'Officer Id'],
    right_on=['OTDATE', 'IDNO6'],
    suffixes=('_massdot', '_court')
)

# Step 2: Filter rows where Military Time is within the specified range
quantum = merged_df[
    (merged_df['Military Start Time'] <= merged_df['Military Time']) &
    (merged_df['Military Time'] <= merged_df['Military End Time'])
][[
    'Officer Id', 'OTDATE', 'Event/Written Date', 
    'Military Start Time', 'Military Time', 'Military End Time', 'Issuing Agency', 'ASSIGNED_DESC'
]]

# Reset the index if needed
quantum = quantum.reset_index(drop=True)

In [71]:
quantum

,Officer Id,OTDATE,Event/Written Date,Military Start Time,Military Time,Military End Time,Issuing Agency,ASSIGNED_DESC
0,10004,2020-01-07,2020-01-07,09:00,09:35,10:00,Boston Police District A-1,E-5 DETECTIVES
1,11449,2020-01-14,2020-01-14,14:00,14:35,15:00,Boston Police District C-6,B-2 DCU SQUAD
2,11449,2020-01-14,2020-01-14,14:00,14:35,15:00,Boston Police District C-6,B-2 DCU SQUAD
3,11449,2020-01-14,2020-01-14,14:00,14:35,15:00,Boston Police District C-6,B-2 DCU SQUAD
4,11413,2020-02-11,2020-02-11,08:30,12:00,12:00,Boston Police District D-4,DISTRICT 04
5,131204,2020-03-04,2020-03-04,08:30,11:57,17:00,Boston Police District B-3,DISTRICT 03
6,108895,2020-03-04,2020-03-04,08:30,08:50,17:00,Boston Police District B-3,DISTRICT 03
7,10801,2020-03-06,2020-03-06,08:30,09:37,10:30,Boston Police District B-3,DISTRICT 11
8,15315,2020-08-18,2020-08-18,09:00,10:24,12:00,Boston Police District D-4,DISTRICT 03
9,8271,2021-02-24,2021-02-24,09:00,09:25,09:30,Boston Police District A-1,A-1 DETECTIVES


In [72]:
# Remove any duplicate rows
quantum = quantum.drop_duplicates()

# Reset the index if needed
quantum = quantum.reset_index(drop=True)

In [73]:
quantum

,Officer Id,OTDATE,Event/Written Date,Military Start Time,Military Time,Military End Time,Issuing Agency,ASSIGNED_DESC
0,10004,2020-01-07,2020-01-07,09:00,09:35,10:00,Boston Police District A-1,E-5 DETECTIVES
1,11449,2020-01-14,2020-01-14,14:00,14:35,15:00,Boston Police District C-6,B-2 DCU SQUAD
2,11413,2020-02-11,2020-02-11,08:30,12:00,12:00,Boston Police District D-4,DISTRICT 04
3,131204,2020-03-04,2020-03-04,08:30,11:57,17:00,Boston Police District B-3,DISTRICT 03
4,108895,2020-03-04,2020-03-04,08:30,08:50,17:00,Boston Police District B-3,DISTRICT 03
5,10801,2020-03-06,2020-03-06,08:30,09:37,10:30,Boston Police District B-3,DISTRICT 11
6,15315,2020-08-18,2020-08-18,09:00,10:24,12:00,Boston Police District D-4,DISTRICT 03
7,8271,2021-02-24,2021-02-24,09:00,09:25,09:30,Boston Police District A-1,A-1 DETECTIVES
8,144325,2021-04-22,2021-04-22,08:30,12:18,14:30,Boston Police District A-7,DISTRICT 07
9,106745,2021-07-20,2021-07-20,14:00,14:20,14:30,Boston Police District E-13,DISTRICT 02


In [74]:
# print all unique values in the 'Issuing Agency' column
quantum['Issuing Agency'].unique()

array(['Boston Police District A-1', 'Boston Police District C-6',
       'Boston Police District D-4', 'Boston Police District B-3',
       'Boston Police District A-7', 'Boston Police District E-13',
       'Boston Police Special OPS', ' '], dtype=object)

In [75]:
quantum['Issuing Agency'] = quantum['Issuing Agency'].str.extract('(\d+)')
quantum['Issuing Agency'] = pd.to_numeric(quantum['Issuing Agency'], errors='coerce').astype('Int64')
quantum

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/var/folders/4y/r6rcgtbn03z_wwr8z4tjksgr0000gn/T/ipykernel_7760/1817264737.py:1: SyntaxWarning: invalid escape sequence '\d'
  quantum['Issuing Agency'] = quantum['Issuing Agency'].str.extract('(\d+)')


,Officer Id,OTDATE,Event/Written Date,Military Start Time,Military Time,Military End Time,Issuing Agency,ASSIGNED_DESC
0,10004,2020-01-07,2020-01-07,09:00,09:35,10:00,1,E-5 DETECTIVES
1,11449,2020-01-14,2020-01-14,14:00,14:35,15:00,6,B-2 DCU SQUAD
2,11413,2020-02-11,2020-02-11,08:30,12:00,12:00,4,DISTRICT 04
3,131204,2020-03-04,2020-03-04,08:30,11:57,17:00,3,DISTRICT 03
4,108895,2020-03-04,2020-03-04,08:30,08:50,17:00,3,DISTRICT 03
5,10801,2020-03-06,2020-03-06,08:30,09:37,10:30,3,DISTRICT 11
6,15315,2020-08-18,2020-08-18,09:00,10:24,12:00,4,DISTRICT 03
7,8271,2021-02-24,2021-02-24,09:00,09:25,09:30,1,A-1 DETECTIVES
8,144325,2021-04-22,2021-04-22,08:30,12:18,14:30,7,DISTRICT 07
9,106745,2021-07-20,2021-07-20,14:00,14:20,14:30,13,DISTRICT 02


In [76]:
# unique ASSIGNED_DESC values in events df
quantum['ASSIGNED_DESC'].unique()

array(['E-5   DETECTIVES', 'B-2\xa0\xa0 DCU SQUAD', 'DISTRICT 04',
       'DISTRICT 03', 'DISTRICT 11', 'A-1   DETECTIVES', 'DISTRICT 07',
       'DISTRICT 02', 'MOBILE OPERATIONS PATROL /SWAT'], dtype=object)

In [77]:
# Define the mapping dictionary with all integer values and set others to 0
mapping = {
    'E-5   DETECTIVES': 0,
    'B-2\xa0\xa0 DCU SQUAD': 0,
    'DISTRICT 04': 4,
    'DISTRICT 03': 3,
    'DISTRICT 11': 11,
    'A-1   DETECTIVES': 0,
    'DISTRICT 07': 7,
    'DISTRICT 02': 2,
    'MOBILE OPERATIONS PATROL /SWAT': 0
}

# Replace the values in the 'ASSIGNED_DESC' column
quantum['ASSIGNED_DESC'] = quantum['ASSIGNED_DESC'].replace(mapping)

# If ASSIGNED_DESC is not an integer, replace it with 0
quantum['ASSIGNED_DESC'] = pd.to_numeric(quantum['ASSIGNED_DESC'], errors='coerce').fillna(0).astype(int)

# Display the updated DataFrame
quantum

/var/folders/4y/r6rcgtbn03z_wwr8z4tjksgr0000gn/T/ipykernel_7760/839237203.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  quantum['ASSIGNED_DESC'] = quantum['ASSIGNED_DESC'].replace(mapping)


,Officer Id,OTDATE,Event/Written Date,Military Start Time,Military Time,Military End Time,Issuing Agency,ASSIGNED_DESC
0,10004,2020-01-07,2020-01-07,09:00,09:35,10:00,1,0
1,11449,2020-01-14,2020-01-14,14:00,14:35,15:00,6,0
2,11413,2020-02-11,2020-02-11,08:30,12:00,12:00,4,4
3,131204,2020-03-04,2020-03-04,08:30,11:57,17:00,3,3
4,108895,2020-03-04,2020-03-04,08:30,08:50,17:00,3,3
5,10801,2020-03-06,2020-03-06,08:30,09:37,10:30,3,11
6,15315,2020-08-18,2020-08-18,09:00,10:24,12:00,4,3
7,8271,2021-02-24,2021-02-24,09:00,09:25,09:30,1,0
8,144325,2021-04-22,2021-04-22,08:30,12:18,14:30,7,7
9,106745,2021-07-20,2021-07-20,14:00,14:20,14:30,13,2


In [78]:
# Percent of issuing agency rows and ASSIGNED_DESC rows that are the same 
same_rows = quantum[quantum['Issuing Agency'] == quantum['ASSIGNED_DESC']]
percent_same_rows = (same_rows.shape[0] / quantum.shape[0]) * 100
percent_same_rows

25.0

In [79]:
# Generate != rows

# Filter out rows where 'Issuing Agency' or 'District_Worked' is None or missing
filtered_quantum = quantum.dropna(subset=['Issuing Agency', 'ASSIGNED_DESC'])

# Drop any rows where 'ASSIGNED_DESC' is 0
filtered_quantum = filtered_quantum[filtered_quantum['ASSIGNED_DESC'] != 0]

# Percent of issuing agency rows and district worked rows that are the diff
diff = filtered_quantum[filtered_quantum['Issuing Agency'] != filtered_quantum['ASSIGNED_DESC']]
percent_diff = len(diff) / len(quantum) * 100
print(percent_diff)
diff

18.75


,Officer Id,OTDATE,Event/Written Date,Military Start Time,Military Time,Military End Time,Issuing Agency,ASSIGNED_DESC
5,10801,2020-03-06,2020-03-06,08:30,09:37,10:30,3,11
6,15315,2020-08-18,2020-08-18,09:00,10:24,12:00,4,3
9,106745,2021-07-20,2021-07-20,14:00,14:20,14:30,13,2


In [83]:
# Generate == rows

# Filter out rows where 'Issuing Agency' or 'District_Worked' is None or missing
filtered_quantum = quantum.dropna(subset=['Issuing Agency', 'ASSIGNED_DESC'])

# Drop any rows where 'ASSIGNED_DESC' is 0
filtered_quantum = filtered_quantum[filtered_quantum['ASSIGNED_DESC'] != 0]

# Percent of issuing agency rows and district worked rows that are the same
same = filtered_quantum[filtered_quantum['Issuing Agency'] == filtered_quantum['ASSIGNED_DESC']]
percent_same = len(same) / len(quantum) * 100
print(percent_same)
same

25.0


,Officer Id,OTDATE,Event/Written Date,Military Start Time,Military Time,Military End Time,Issuing Agency,ASSIGNED_DESC
2,11413,2020-02-11,2020-02-11,08:30,12:00,12:00,4,4
3,131204,2020-03-04,2020-03-04,08:30,11:57,17:00,3,3
4,108895,2020-03-04,2020-03-04,08:30,08:50,17:00,3,3
8,144325,2021-04-22,2021-04-22,08:30,12:18,14:30,7,7


In [85]:
# Unknown comparisons
print(100 - percent_diff - percent_same)

56.25
